In [1]:
import com.microsoft.spark.fabric


# Build the Gold dimension from the Silver Lakehouse table
gold_df = spark.sql("""
SELECT
    UPPER(TRIM(airport_icao)) AS airport_icao,
    UPPER(TRIM(airport_iata)) AS iata_code,
    TRIM(airport_name) AS airport_name,
    TRIM(airport_type) AS airport_type,
    TRIM(city) AS city,
    TRIM(country) AS country,
    CAST(latitude AS DOUBLE) AS latitude,
    CAST(longitude AS DOUBLE) AS longitude,
    CAST(icao_is_official AS BOOLEAN) AS icao_is_official,
    valid_from,
    valid_to,
    is_current
FROM EAA_WorkSpace.EAA_LakeHouse.silver.airport
""")


# Optional: write a Delta copy to the Lakehouse Gold schema
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("EAA_WorkSpace.EAA_LakeHouse.gold.dim_airport")
)


# Write the same DataFrame to the EAA_Gold Warehouse
(
    gold_df.write
    .mode("overwrite")
    .synapsesql("EAA_Gold.dbo.dim_airport")
)

print("Gold Lakehouse and Warehouse tables written successfully.")

StatementMeta(, c07f010c-b938-40f2-9af3-113a93ba48fa, 3, Finished, Available, Finished, False)

Gold Lakehouse and Warehouse tables written successfully.


In [3]:
#### VALIDATION ####

print(f"Rows: {gold_df.count():,}")

gold_df.printSchema()

display(gold_df.limit(10))

StatementMeta(, c07f010c-b938-40f2-9af3-113a93ba48fa, 5, Finished, Available, Finished, False)

Rows: 3,186
root
 |-- airport_icao: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- airport_name: string (nullable = true)
 |-- airport_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- icao_is_official: boolean (nullable = true)
 |-- valid_from: date (nullable = true)
 |-- valid_to: date (nullable = true)
 |-- is_current: boolean (nullable = true)



SynapseWidget(Synapse.DataFrame, 11637e8b-1c76-47ee-ad65-4b10c7343a6a)

In [6]:
# Row count and airport coverage

spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT airport_icao) AS distinct_airports,
    MIN(valid_from) AS earliest_valid_from,
    MAX(valid_from) AS latest_valid_from
FROM EAA_WorkSpace.EAA_LakeHouse.gold.dim_airport
""").show()

StatementMeta(, c07f010c-b938-40f2-9af3-113a93ba48fa, 8, Finished, Available, Finished, False)

+----------+-----------------+-------------------+-----------------+
|total_rows|distinct_airports|earliest_valid_from|latest_valid_from|
+----------+-----------------+-------------------+-----------------+
|      3186|             3186|         2026-08-05|       2026-08-05|
+----------+-----------------+-------------------+-----------------+



In [7]:
spark.sql("""
SELECT *
FROM EAA_WorkSpace.EAA_LakeHouse.gold.dim_airport
WHERE airport_icao IS NULL
""").show(truncate=False)

StatementMeta(, c07f010c-b938-40f2-9af3-113a93ba48fa, 9, Finished, Available, Finished, False)

+------------+---------+------------+------------+----+-------+--------+---------+----------------+----------+--------+----------+
|airport_icao|iata_code|airport_name|airport_type|city|country|latitude|longitude|icao_is_official|valid_from|valid_to|is_current|
+------------+---------+------------+------------+----+-------+--------+---------+----------------+----------+--------+----------+
+------------+---------+------------+------------+----+-------+--------+---------+----------------+----------+--------+----------+



In [4]:
duplicates = (
    gold_df
    .groupBy("airport_icao", "valid_from")
    .count()
    .filter("count > 1")
)

if duplicates.count() > 0:
    print("WARNING: Duplicate airport versions found.")
    display(duplicates)
else:
    print("No duplicate airport versions found.")

StatementMeta(, c07f010c-b938-40f2-9af3-113a93ba48fa, 6, Finished, Available, Finished, False)

No duplicate airport versions found.


In [ ]:
spark.sql("""
SELECT
    airport_icao,
    COUNT(*) AS current_row_count
FROM gold.dim_airport
WHERE is_current = true
GROUP BY airport_icao
HAVING COUNT(*) > 1
ORDER BY current_row_count DESC
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    airport_icao,
    airport_name
FROM gold.dim_airport
WHERE airport_icao IS NULL
   OR LENGTH(airport_icao) <> 4
   OR airport_icao NOT RLIKE '^[A-Z]{4}$'
""").show(truncate=False)